In [1]:
import csv
import re
from pathlib import Path
from datetime import datetime

In [2]:
REQUIRED_FIELDS = {
    "customers.csv": [
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ],

    "orders.csv": [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp"
    ],

    "order_items.csv": [
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value"
    ],

    "products.csv": [
        "product_id",
        "product_category_name"
    ],

    "sellers.csv": [
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    ],

    "order_payments.csv": [
        "order_id",
        "payment_type",
        "payment_value"
    ],

    "order_reviews.csv": [
        "review_id",
        "order_id",
        "review_score"
    ],

    "geolocation.csv": [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    ],

    "product_category_name_translation.csv": [
        "product_category_name",
        "product_category_name_english"
    ]
}

In [ ]:
#task1
for filename, fields in REQUIRED_FIELDS.items():
    print(f"{filename}: {fields}")

customers.csv: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
orders.csv: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp']
order_items.csv: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'price', 'freight_value']
products.csv: ['product_id', 'product_category_name']
sellers.csv: ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']
order_payments.csv: ['order_id', 'payment_type', 'payment_value']
order_reviews.csv: ['review_id', 'order_id', 'review_score']
geolocation.csv: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']
product_category_name_translation.csv: ['product_category_name', 'product_category_name_english']


In [ ]:
#task2
def validate_rows(data, filename):
    valid = []
    rejected = []

    for row_number, row in enumerate(data, start=2):
        errors = []

        for field in REQUIRED_FIELDS[filename]:
            if not row.get(field):
                errors.append(f"Missing {field}")

        if errors:
            rejected.append({
                "row": row_number,
                "data": row,
                "reason": ", ".join(errors)
            })
        else:
            valid.append(row)

    return valid, rejected

In [5]:
sample_data = [
    {"customer_id": "123", "customer_unique_id": "456",
     "customer_zip_code_prefix": "60000", "customer_city": "Chennai",
     "customer_state": "TN"},

    {"customer_id": "", "customer_unique_id": "789",
     "customer_zip_code_prefix": "60001", "customer_city": "Coimbatore",
     "customer_state": "TN"}
]

valid, rejected = validate_rows(sample_data, "customers.csv")

print("Valid:", valid)
print("Rejected:", rejected)

Valid: [{'customer_id': '123', 'customer_unique_id': '456', 'customer_zip_code_prefix': '60000', 'customer_city': 'Chennai', 'customer_state': 'TN'}]
Rejected: [{'row': 3, 'data': {'customer_id': '', 'customer_unique_id': '789', 'customer_zip_code_prefix': '60001', 'customer_city': 'Coimbatore', 'customer_state': 'TN'}, 'reason': 'Missing customer_id'}]


In [6]:
#task3
import csv

def save_bad_records(rejected):
    with open("bad_records.csv", "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=["row", "data", "reason"])
        writer.writeheader()
        writer.writerows(rejected)

In [7]:
save_bad_records(rejected)
with open("bad_records.csv", "r", encoding="utf-8") as file:
    print(file.read())

row,data,reason
3,"{'customer_id': '', 'customer_unique_id': '789', 'customer_zip_code_prefix': '60001', 'customer_city': 'Coimbatore', 'customer_state': 'TN'}",Missing customer_id



In [8]:
#task4
def check_types(row):
    errors = []

    try:
        float(row["price"])
    except (ValueError, KeyError):
        errors.append("Invalid price")

    try:
        datetime.fromisoformat(row["order_purchase_timestamp"])
    except (ValueError, KeyError):
        errors.append("Invalid date")

    return errors

In [10]:
sample = {
    "price": "abc",
    "order_purchase_timestamp": "hello"
}

print(check_types(sample))

['Invalid price', 'Invalid date']


In [11]:
#task5
ALLOWED_ORDER_STATUSES = [
    "created", "approved", "invoiced", "processing",
    "shipped", "delivered", "canceled", "unavailable"
]

def check_business_rules(row):
    errors = []

    if "price" in row:
        if float(row["price"]) <= 0:
            errors.append("Price must be greater than 0")

    if "order_status" in row:
        if row["order_status"] not in ALLOWED_ORDER_STATUSES:
            errors.append("Invalid order status")

    return errors

In [12]:
sample = {
    "price": "-10",
    "order_status": "unknown"
}

print(check_business_rules(sample))

['Price must be greater than 0', 'Invalid order status']


In [ ]:
#task6
import re

zip_pattern = r"^\d{5}$"
id_pattern = r"^[0-9a-fA-F]{32}$"
def check_regex(row):
    errors = []

    for field in ["customer_zip_code_prefix", "seller_zip_code_prefix"]:
        if field in row and not re.fullmatch(zip_pattern, row[field]):
            errors.append(f"Invalid {field}")

    for field in ["customer_id", "order_id", "product_id", "seller_id"]:
        if field in row and not re.fullmatch(id_pattern, row[field]):
            errors.append(f"Invalid {field}")

    return errors

In [14]:
sample = {
    "customer_zip_code_prefix": "123",
    "customer_id": "abc123"
}

print(check_regex(sample))

['Invalid customer_zip_code_prefix', 'Invalid customer_id']


In [15]:
#task7
class SchemaError(Exception):
    pass
class FileFormatError(Exception):
    pass

try:
    raise SchemaError("Missing required columns")
except SchemaError as e:
    print(e)



Missing required columns


In [16]:
try:
    raise FileFormatError("Invalid CSV file")
except FileFormatError as e:
    print(e)

Invalid CSV file


In [17]:
#task8
from pathlib import Path

def check_file(filepath):
    filepath = Path(filepath)

    if not filepath.exists():
        raise FileFormatError("File not found")

    if filepath.stat().st_size == 0:
        raise FileFormatError("File is empty")

    print("File is ready to process")

In [18]:
try:
    check_file("missing.csv")
except FileFormatError as e:
    print(e)

File not found


In [20]:
#task9
def print_summary(data, valid, rejected):
    print("Total rows:", len(data))
    print("Valid rows:", len(valid))
    print("Rejected rows:", len(rejected))
print_summary(sample_data, valid, rejected)

Total rows: 2
Valid rows: 1
Rejected rows: 1


In [29]:
#task10
def check_rejection_threshold(total, rejected, threshold=0.20):
    if total == 0:
        return

    rejection_rate = rejected / total

    if rejection_rate > threshold:
        raise FileFormatError("Too many rejected rows")

    print("Rejection rate acceptable")

In [30]:
try:
    check_rejection_threshold(100, 25)
except FileFormatError as e:
    print(e)

Too many rejected rows
